# Tool Definition

In [1]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [2]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [4]:
tool1.invoke({"x": 467})

21.61018278497431

# Adding to agents

In [10]:
from langchain.agents import create_agent

from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(
    model="qwen3.5"
)

agent = create_agent(
    model=model,
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately **21.6102**.


In [12]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='460e0b2c-8e9d-4980-9c0b-5ea38fe3e785'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-02T17:21:46.513346Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4855073600, 'load_duration': 3915001400, 'prompt_eval_count': 301, 'prompt_eval_duration': 120953600, 'eval_count': 74, 'eval_duration': 718187600, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d4f37-0f19-78c3-8ea9-e9c6923b0790-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': '0b603f49-e68e-4fba-b207-77522462f050', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 301, 'output_tokens': 74, 'total_tokens': 375}),
 ToolMessage(content='21.61018278497431', name='square_root', id='5e2be272-abab-4cdd-9c41-4d07dbe028cc', tool_call_id='0b603f49-e68e-4fba-b207-77522462f050'),
 AIMessage

In [13]:
print(response['messages'][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': '0b603f49-e68e-4fba-b207-77522462f050', 'type': 'tool_call'}]


# Web Search tool

## Wthout tool

In [14]:
question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke({'messages': [question]})

print(response['messages'][-1].content)

I'm an AI assistant with training knowledge up to December 2024. My knowledge is based on information available up to that point, which means I have been trained on data and information from that time period.

While I can't provide the most real-time news or events after my knowledge cutoff, I can:

- Answer questions about topics that were current up to my training date
- Help with calculations and reasoning based on that knowledge
- Provide general information on various subjects
- Assist with writing, analysis, and other tasks

If you need information about recent events or developments after December 2024, I'd recommend checking the latest sources. Is there something specific you'd like help with?


## Add Web search tool

### Duckduck go search

- We will use the langchain search integrations. : https://docs.langchain.com/oss/python/integrations/providers/duckduckgo_search

How to install
`pip install ddgs` or `uv add ddgs`

inorder to import ddgs you will also need to add langchain-community

- usage examples: https://docs.langchain.com/oss/python/integrations/tools/ddg



In [16]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

res = search.invoke("Who is Najeeb Arif?")

pprint(res)

('The latest victim of journalistic assassinationisNajeebAhmed, the JNU '
 'studentwhodisappeared from the campus on 15th October, 2016, after an ... '
 'User profile ofNajeebArifon Hugging Face ...NajeebArif... NajeebArif Houston '
 'Tex, USA: A reception and meet-and-greet was hosted byNajeebAhmad, a '
 'respected business leader, in honor of Kaukab Iqbal, Chairman of the ... It '
 'was the wayArifstill believed in things: in ...isa former journalist whose '
 'storytelling journey spans India, the Middle East, and Australia. '
 'ArifAlviiscurrently attending the FDI Annual World Dental Congress in '
 'Istanbul- Turkey, where he will be concluding his second tenure as an ...')


To get more additional information (e.g. link, source) use DuckDuckGoSearchResults()

In [17]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()

res = search.invoke("Najeeb Arif")

pprint(res)

('snippet: Read writing fromNajeebArifon Medium.May 31, 2022.NajeebArif. 129 '
 'followers., title: NajeebArif– Medium, link: https://najeebarif.medium.com/, '
 'snippet: Sumary.NajeebArif. Innovative and deadline-driven full stack '
 'developer with 5+ years of experience designing and developing microservices '
 "based cloud native backend services deployed..., title: Welcome toNajeeb's "
 'website., link: https://najeebarif.github.io/, snippet: ViewNajeebArif’s '
 'profile on LinkedIn, a professional community of 1 billion members.View '
 'mutual connections withNajeeb.Najeebcan introduce you to 10+ people at IBM., '
 'title: NajeebArif- IBM | LinkedIn, link: '
 'https://in.linkedin.com/in/najeeb-arif, snippet: @najeeb_arif1. AI Chatbot '
 'Developer and Business Automation Expert. Pakistan. English, Hindi. About '
 "me. Hey, I'mNajeeb, an AI chatbot and automation Expert., title: NajeebA | "
 'Profile | Fiverr, link: https://www.fiverr.com/najeeb_arif1')


By default the results are returned as a comma-separated string of key-value pairs from the original search results. You can also choose to return the search results as a list by setting output_format="list" or as a JSON string by setting output_format="json".

In [19]:
search = DuckDuckGoSearchResults(output_format="list")

search.invoke("Najeeb Arif")

[{'snippet': 'This is the list of serving officers in the Pakistan Army. At present the Army has one Field Marshal, 30 Lieutenant Generals (including one from Army Medical Corps) and 186 Major Generals (including 28 from Army Medical Corps).',
  'title': 'List of serving generals of the Pakistan Army - Wikipedia',
  'link': 'https://en.wikipedia.org/wiki/List_of_serving_generals_of_the_Pakistan_Army'},
 {'snippet': "NajeebArif. 2,585 likes · 8 talking about this. Certified eBay Expert eBay growth Consultant Click for eBay service's Wa.Me/+923077714208",
  'title': 'Najeeb Arif - Facebook',
  'link': 'https://www.facebook.com/najeeb.arif.90/'},
 {'snippet': '"Nice write-up" is published byNajeebArif.',
  'title': 'Nice write-up - Najeeb Arif - Medium',
  'link': 'https://najeebarif.medium.com/nice-write-up-fb9578cddee5'},
 {'snippet': 'NajeebS.A. is a former journalist whose storytelling journey spans India, the Middle East, and Australia. His short fiction has appeared in multiple anth

You can also just search for news articles. Use the keyword backend="news"

In [20]:
search = DuckDuckGoSearchResults(backend="news")

search.invoke("Najeeb Arif")

'snippet: ISLAMABAD: With the approval of Prime Minister Shehbaz Sharif, the government on Friday notified the appointment of Dr Najeeb Ahmad Memon as the first director general of the Tax Policy Office (TPO) in the Finance Division on a Special Professional Pay ..., title: Dr Najeeb appointed first DG of Tax Policy Office, link: https://www.thenews.com.pk/latest/1353436-dr-najeeb-appointed-first-dg-of-tax-policy-office, date: 2025-10-25T00:00:00+00:00, source: The News International, snippet: SRINAGAR: The Government of Jammu and Kashmir on Wednesday ordered a major reshuffle in the prosecution department, ..., title: 104 Officers Appointed as Public Prosecutors in Jammu Kashmir, link: https://kashmirlife.net/104-officers-appointed-as-public-prosecutors-in-jammu-kashmir-430604/, date: 2026-04-01T17:38:07+00:00, source: Kashmir Life, snippet: Know all about Maharashtra Assembly Election Constituency Mumbadevi Results, candidate list, last winner, winner name, runner up and more ..., ti

You can also directly pass a custom DuckDuckGoSearchAPIWrapper to DuckDuckGoSearchResults to provide more control over the search results.

In [21]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper(region="de-de", time="d", max_results=2)

search = DuckDuckGoSearchResults(api_wrapper=wrapper, source="news")

search.invoke("Obama")

"snippet: Barack Hussein Obama II [bəˈɹɑːk hʊˈseɪn oʊˈbɑːmə] (* 4. August 1961 in Honolulu, Hawaii) ist ein US-amerikanischer Politiker der Demokratischen Partei. Er war von 2009 bis 2017 der 44. Präsident der Vereinigten Staaten.Obama ist ein auf US-Verfassungsrecht spezialisierter Rechtsanwalt. 1992 schloss er sich der Demokratischen Partei an, für die er 1997 Mitglied im Senat von Illinois wurde. Im Anschluss gehörte er von 2005 bis 2008 als Junior Senator für diesen US-Bundesstaat dem Senat der Vereinigten Staaten an. In den Vorwahlen zur Präsidentschaftswahl in den Vereinigten Staaten 2008 konnte er sich parteiintern äußerst knapp gegen Hillary Clinton durchsetzen. Die Präsidentschaftswahl des Jahres 2008 gewann er dann als Kandidat der Demokraten gegen den Republikaner John McCain.Mit seinem Einzug in das Weiße Haus im Januar 2009 bekleidete erstmals ein Afroamerikaner das Amt des Präsidenten. Bei der Wahl des Jahres 2012 setzte sich Obama gegenüber seinem republikanischen Heraus

## Adding the search tool to the agent

In [22]:
from typing import Dict, Any

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return search.invoke(query)

web_search.invoke("Who is the current mayor of San Francisco?")


"ToggleMayorofSanFranciscosubsection. 4.1 Policy priorities and actions in office.Daniel Lawrence Lurie[1] (born February 4, 1977) is an American politician and philanthropistwhoisthe46th and incumbentmayorofSanFrancisco, serving since 2025. SANFRANCISCO(KGO) -- Change in leadership has come toSanFranciscoas Daniel Lurie was sworn in as the city's newmayoron Wednesday. SanFrancisco’s 46thmayorhoned his approach to tackling poverty while working at New York’s Robin Hood Foundation and has introduced it to the West Coast through the Tipping Point Community. CanSanFrancisco’s NewMayorMake the City Shine Again?Heather KnightistheSanFranciscobureau chief. Daniel Lurie willbethefifthSanFranciscomayorshe has covered. “SanFranciscohas long been known for its values of tolerance and inclusion,” he said. “But nothing about those values instructs us to allow nearly 8,000 people to experience homelessness in our city.” The crowd watches asMayorDaniel Lurie deliver his inaugural address."

In [24]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of New York?")

response = agent.invoke({'messages': [question]})

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of New York?', additional_kwargs={}, response_metadata={}, id='3081723d-c5d9-46f6-933c-b28ef9edc88c'),
 AIMessage(content='The current mayor of New York City is **Eric Adams**. He is a Democrat who was inaugurated as the 109th mayor on January 1, 2022. Adams succeeded Bill de Blasio, serving one term after winning the mayoral election.', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-02T17:44:45.6492412Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1655696900, 'load_duration': 116245900, 'prompt_eval_count': 276, 'prompt_eval_duration': 366621700, 'eval_count': 113, 'eval_duration': 1114221400, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d4f4c-26d8-7d21-b92f-8e71d85e22f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 113, 'total_tokens': 389})]
